# **image_detection**

In [308]:
import cv2 as cv
import numpy as np

In [309]:
yolo_path="/content/yolov3 (1).weights"
yolo_config="/content/yolov3 (1).cfg"
image=cv.imread("/content/city_image.jpg")
height, width, channel=image.shape

In [310]:
width

612

In [311]:
classes=[]
with open('/content/coco (1).names', 'r') as f:
  classes=[line.strip() for line in f.readlines()]

In [312]:
classes

['person',
 'bicycle',
 'car',
 'motorbike',
 'aeroplane',
 'bus',
 'train',
 'truck',
 'boat',
 'traffic light',
 'fire hydrant',
 'stop sign',
 'parking meter',
 'bench',
 'bird',
 'cat',
 'dog',
 'horse',
 'sheep',
 'cow',
 'elephant',
 'bear',
 'zebra',
 'giraffe',
 'backpack',
 'umbrella',
 'handbag',
 'tie',
 'suitcase',
 'frisbee',
 'skis',
 'snowboard',
 'sports ball',
 'kite',
 'baseball bat',
 'baseball glove',
 'skateboard',
 'surfboard',
 'tennis racket',
 'bottle',
 'wine glass',
 'cup',
 'fork',
 'knife',
 'spoon',
 'bowl',
 'banana',
 'apple',
 'sandwich',
 'orange',
 'broccoli',
 'carrot',
 'hot dog',
 'pizza',
 'donut',
 'cake',
 'chair',
 'sofa',
 'pottedplant',
 'bed',
 'diningtable',
 'toilet',
 'tvmonitor',
 'laptop',
 'mouse',
 'remote',
 'keyboard',
 'cell phone',
 'microwave',
 'oven',
 'toaster',
 'sink',
 'refrigerator',
 'book',
 'clock',
 'vase',
 'scissors',
 'teddy bear',
 'hair drier',
 'toothbrush']

In [313]:
yolo_net=cv.dnn.readNet(yolo_config, yolo_path)

In [314]:
layers=yolo_net.getLayerNames()
output_layers=[layers[i-1] for i in yolo_net.getUnconnectedOutLayers()]

In [315]:
output_layers

['yolo_82', 'yolo_94', 'yolo_106']

In [316]:
image_blob=cv.dnn.blobFromImage(image, 0.00329,  (416, 416), (0, 0, 0), True, crop=False)

In [317]:
yolo_net.setInput(image_blob)
outputs=yolo_net.forward(output_layers)

In [318]:
outputs[0].shape

(507, 85)

In [319]:
class_ids = []
confidences = []
boxes = []
for out in outputs:
    for detection in out:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]

        # Object detected
        # if confidence > 0.6:
        center_x = int(detection[0] * width)
        center_y = int(detection[1] * height)
        w = int(detection[2] * width)
        h = int(detection[3] * height)

        # Rectangle coordinates
        x = int(center_x - w / 2)
        y = int(center_y - h / 2)
        boxes.append([x, y, w, h])
        confidences.append(float(confidence))
        class_ids.append(class_id)

In [320]:
colors=np.random.uniform(0, 255, size=(len(classes), 3))
# colors.shape
len(boxes)

10647

In [321]:
colors=np.random.uniform(0, 255, size=(len(classes), 3))
indexes = cv.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)
for i in range(len(boxes)):
      if i in indexes:
        x, y, w, h = boxes[i]
        label = str(classes[class_ids[i]])
        color = colors[class_ids[i]]
        cv.rectangle(image, (x, y), (x + w, y + h), color, 1)
        cv.putText(image, label, (x, y + 15), cv.FONT_HERSHEY_COMPLEX_SMALL, 1, color, 1)


cv.imwrite('/content/city_image_detected.jpg', image)

True

## video_detection

In [326]:
cap = cv.VideoCapture('/content/traffic-mini (1).mp4')
video_cod = cv.VideoWriter_fourcc(*'XVID')
video_output = cv.VideoWriter('driving_camera_det_yolov3_cv.mp4',
                      video_cod,
                      10,
                      (1280,720))

In [327]:
while (cap.isOpened()):
  ret, img = cap.read()
  if ret == True:
            #img = cv.resize(frame, None, fx=0.8, fy=0.8)
        height, width, channels = img.shape
        #print(height, width)
        blob = cv.dnn.blobFromImage(img, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
        yolo_net1.setInput(blob)
        outs = yolo_net.forward(output_layers)
        class_ids = []
        confidences = []
        boxes = []
        for out in outs:
            for detection in out:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                if confidence > 0.5:
                    # Object detected
                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)
                    w = int(detection[2] * width)
                    h = int(detection[3] * height)

                    # Rectangle coordinates
                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)

                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)

        indexes = cv.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)
        font = cv.FONT_HERSHEY_COMPLEX_SMALL
        for i in range(len(boxes)):
            if i in indexes:
                x, y, w, h = boxes[i]
                label = str(classes[class_ids[i]])
                color = colors[class_ids[i]]
                cv.rectangle(img, (x, y), (x + w, y + h), color, 1)
                cv.putText(img, label, (x, y + 15), font, 1, color, 1)
        # cv.imshow('Frame',img)
        video_output.write(img)

        if cv.waitKey(25) & 0xFF == ord('q'):
            break
        else:
            break

cap.release()
video_output.release()
cv.destroyAllWindows()